In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import random
from lorem_text import lorem

In [ ]:
# Função auxiliar para pegar palavras únicas que formarão o conteúdo
# dos campos nomes, empresas e produtos
def pegar_unicas(qtd):
    palavras = set()
    while len(palavras) < qtd:
        palavras.update(lorem.sentence().split())
    return random.sample(list(palavras), qtd)

In [ ]:
# Cria listas sem repetições
nomes = pegar_unicas(15)
empresas = pegar_unicas(20)
produtos = pegar_unicas(25)

print("Nomes:", nomes)
print("Empresas:", empresas)
print("Produtos:", produtos)

In [ ]:
# Dataset
n = 200_000
np.random.seed(42)
df = pd.DataFrame({
    'nome': np.random.choice(nomes, n),
    'empresa': np.random.choice(empresas, n),
    'produto': np.random.choice(produtos, n),
    'valor': np.round(np.random.uniform(0.01, 10, n), 2)
})
df.info()

In [ ]:
# Buscas
n = 2_000
df_buscas = pd.DataFrame({
    'nome': np.random.choice(nomes, n),
    'empresa': np.random.choice(empresas, n),
    'produto': np.random.choice(produtos, n),
})
df_buscas.info()

In [ ]:
# Função que pega a soma dos valores do dataset para cada busca, usando filtros
def get_soma(df, df_buscas):
    soma = 0
    t_ini = time.time()
    for _, busca in df_buscas.iterrows():
        soma += df[
            (df['nome'] == busca['nome']) &
            (df['empresa'] == busca['empresa']) &
            (df['produto'] == busca['produto'])
        ]['valor'].sum()
    t_total = time.time() - t_ini
    return soma, t_total

In [ ]:
# 1. string
soma, t_string = get_soma(df, df_buscas)
print(f"Soma: {soma:.2f}")

In [ ]:
# 2. string vectorizado
t_ini = time.time()
soma = df.merge(df_buscas, on=['nome', 'empresa', 'produto'])['valor'].sum()
t_string_vectorized = time.time() - t_ini
print(f"Soma: {soma:.2f}")

In [ ]:
# Altera os campos para o tipo category
df['nome'] = df['nome'].astype('category')
df['empresa'] = df['empresa'].astype('category')
df['produto'] = df['produto'].astype('category')

In [ ]:
# 3. category
soma, t_category = get_soma(df, df_buscas)
print(f"Soma: {soma:.2f}")

In [ ]:
# 4. category vectorizado
t_ini = time.time()
soma = df.merge(df_buscas, on=['nome', 'empresa', 'produto'])['valor'].sum()
t_category_vectorized = time.time() - t_ini
print(f"Soma: {soma:.2f}")

In [ ]:
# Gráfico dos tempos do cálculo usando filtros
labels = ["string", "category"]
tempos = [t_string, t_category]

plt.figure(figsize=(8, 4))
plt.bar(labels, tempos)
plt.ylabel("Tempo (segundos)")
plt.title(
    f"Comparação de performance - Pandas\n"
    f"Somando o resultado de {f'{len(df_buscas):_}'.replace('_', '.')} buscas"
)

for i, v in enumerate(tempos):
    plt.text(i, v, f"{v:.3f}s", ha="center", va="bottom")

plt.show()

In [ ]:
# Gráfico dos tempos do cálculo usando abordagem vetorizada
labels = ["string vectorizado", "category vectorizado"]
tempos = [t_string_vectorized, t_category_vectorized]

plt.figure(figsize=(8, 4))
plt.bar(labels, tempos)
plt.ylabel("Tempo (segundos)")
plt.title(
    f"Comparação de performance - Pandas\n"
    f"Somando de forma vetorizada o resultado de {f'{len(df_buscas):_}'.replace('_', '.')} buscas"
)

for i, v in enumerate(tempos):
    plt.text(i, v, f"{v:.3f}s", ha="center", va="bottom")

plt.show()